In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from utils import Helper
import urllib3
import warnings
from pathlib import Path
# Suppress the warning
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

utils = Helper()

url = "https://report.pmayg.dord.gov.in/netiay/DataAnalytics/PhysicalProgressRpt.aspx"
session = requests.Session()

def get_hidden_fields(soup):
    fields = {}
    for tag in soup.find_all("input", type="hidden"):
        if tag.get("name"):
            fields[tag["name"]] = tag.get("value", "")
    return fields

def parse_panchayat_table(html, state_name, district_name, block_name, panchayat_name, year_code):
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table", {"id": "ctl00_ContentPlaceHolder1_gvData"})
    if not table:
        # return empty DataFrame with just metadata columns
        return pd.DataFrame(columns=[
            "State","District","Block","Panchayat","Year"
        ])

    df = pd.read_html(str(table))[0]  # keep whatever headers Pandas detects

    # Add metadata
    df["State"] = state_name
    df["District"] = district_name
    df["Block"] = block_name
    df["Panchayat"] = panchayat_name
    df["Year"] = year_code
    return df

def fetch_panchayat_table(year_code, state_code, district_code, block_code, panchayat_code,
                          state_name, district_name, block_name, panchayat_name):
    # Step 1: GET fresh page
    resp = session.get(url, verify=False)
    soup = BeautifulSoup(resp.text, "html.parser")
    hidden = get_hidden_fields(soup)

    # Step 2: select district
    payload_district = {
        **hidden,
        "ctl00$ContentPlaceHolder1$ddlFinYear": year_code,
        "ctl00$ContentPlaceHolder1$ddlState": state_code,
        "ctl00$ContentPlaceHolder1$ddlDistrict": district_code,
        "__EVENTTARGET": "ctl00$ContentPlaceHolder1$ddlDistrict"
    }
    resp2 = session.post(url, data=payload_district, verify=False)
    hidden2 = get_hidden_fields(BeautifulSoup(resp2.text, "html.parser"))

    # Step 3: select block
    payload_block = {
        **hidden2,
        "ctl00$ContentPlaceHolder1$ddlFinYear": year_code,
        "ctl00$ContentPlaceHolder1$ddlState": state_code,
        "ctl00$ContentPlaceHolder1$ddlDistrict": district_code,
        "ctl00$ContentPlaceHolder1$ddlBlock": block_code,
        "__EVENTTARGET": "ctl00$ContentPlaceHolder1$ddlBlock"
    }
    resp3 = session.post(url, data=payload_block, verify=False)
    hidden3 = get_hidden_fields(BeautifulSoup(resp3.text, "html.parser"))

    # Step 4: select panchayat
    payload_panchayat = {
        **hidden3,
        "ctl00$ContentPlaceHolder1$ddlFinYear": year_code,
        "ctl00$ContentPlaceHolder1$ddlState": state_code,
        "ctl00$ContentPlaceHolder1$ddlDistrict": district_code,
        "ctl00$ContentPlaceHolder1$ddlBlock": block_code,
        "ctl00$ContentPlaceHolder1$ddlPanchayat": panchayat_code,
        "__EVENTTARGET": "ctl00$ContentPlaceHolder1$ddlPanchayat",
        "__EVENTARGUMENT": "",
        "__LASTFOCUS": ""
    }
    resp4 = session.post(url, data=payload_panchayat, verify=False)
    hidden4 = get_hidden_fields(BeautifulSoup(resp4.text, "html.parser"))

    # Step 5: submit
    payload_submit = {
        **hidden4,
        "ctl00$ContentPlaceHolder1$ddlFinYear": year_code,
        "ctl00$ContentPlaceHolder1$ddlState": state_code,
        "ctl00$ContentPlaceHolder1$ddlDistrict": district_code,
        "ctl00$ContentPlaceHolder1$ddlBlock": block_code,
        "ctl00$ContentPlaceHolder1$ddlPanchayat": panchayat_code,
        "__EVENTTARGET": "ctl00$ContentPlaceHolder1$btnSubmit",
        "__EVENTARGUMENT": "",
        "__LASTFOCUS": ""
    }
    resp5 = session.post(url, data=payload_submit, verify=False)
    df = parse_panchayat_table(resp5.text,
                            state_name, district_name,
                            block_name, panchayat_name,
                            year_code)


    Ahead = ["Year","State","District","Block"]
    col_order = Ahead + [ col for col in df.columns if col not in Ahead]
    
    new_df = df[col_order]
    return new_df   

def fetch_block_level_data(year_code, state_code, district_code, block_code,
                           state_name, district_name, block_name):
    # Step 1: GET page
    resp = session.get(url, verify=False)
    hidden = get_hidden_fields(BeautifulSoup(resp.text, "html.parser"))

    # Step 2: Select district
    payload_district = {
        **hidden,
        "ctl00$ContentPlaceHolder1$ddlFinYear": year_code,
        "ctl00$ContentPlaceHolder1$ddlState": state_code,
        "ctl00$ContentPlaceHolder1$ddlDistrict": district_code,
        "__EVENTTARGET": "ctl00$ContentPlaceHolder1$ddlDistrict",
        "__EVENTARGUMENT": "",
        "__LASTFOCUS": ""
    }
    resp2 = session.post(url, data=payload_district, verify=False)
    hidden2 = get_hidden_fields(BeautifulSoup(resp2.text, "html.parser"))

    # Step 3: Select block
    payload_block = {
        **hidden2,
        "ctl00$ContentPlaceHolder1$ddlFinYear": year_code,
        "ctl00$ContentPlaceHolder1$ddlState": state_code,
        "ctl00$ContentPlaceHolder1$ddlDistrict": district_code,
        "ctl00$ContentPlaceHolder1$ddlBlock": block_code,
        "__EVENTTARGET": "ctl00$ContentPlaceHolder1$ddlBlock",
        "__EVENTARGUMENT": "",
        "__LASTFOCUS": ""
    }
    resp3 = session.post(url, data=payload_block, verify=False)
    hidden3 = get_hidden_fields(BeautifulSoup(resp3.text, "html.parser"))

    # Step 4: Trigger panchayat dropdown with "0"
    payload_panchayat = {
        **hidden3,
        "ctl00$ContentPlaceHolder1$ddlFinYear": year_code,
        "ctl00$ContentPlaceHolder1$ddlState": state_code,
        "ctl00$ContentPlaceHolder1$ddlDistrict": district_code,
        "ctl00$ContentPlaceHolder1$ddlBlock": block_code,
        "ctl00$ContentPlaceHolder1$ddlPanchayat": "0",
        "__EVENTTARGET": "ctl00$ContentPlaceHolder1$ddlPanchayat",
        "__EVENTARGUMENT": "",
        "__LASTFOCUS": ""
    }
    resp4 = session.post(url, data=payload_panchayat, verify=False)
    hidden4 = get_hidden_fields(BeautifulSoup(resp4.text, "html.parser"))

    # Step 5: Submit
    payload_submit = {
        **hidden4,
        "ctl00$ContentPlaceHolder1$ddlFinYear": year_code,
        "ctl00$ContentPlaceHolder1$ddlState": state_code,
        "ctl00$ContentPlaceHolder1$ddlDistrict": district_code,
        "ctl00$ContentPlaceHolder1$ddlBlock": block_code,
        "ctl00$ContentPlaceHolder1$ddlPanchayat": "0",
        "__EVENTTARGET": "ctl00$ContentPlaceHolder1$btnSubmit",
        "__EVENTARGUMENT": "",
        "__LASTFOCUS": ""
    }
    resp5 = session.post(url, data=payload_submit, verify=False)

    return parse_block_table(resp5.text, state_name, district_name, block_name, year_code)

def parse_block_table(html, state_name, district_name, block_name, year_code):
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table", {"id": "ctl00_ContentPlaceHolder1_gvData"})
    if not table:
        return pd.DataFrame(columns=["Year","State","District","Block"])

    # Read the table
    df = pd.read_html(str(table))[0]

    # Add metadata
    df["Year"] = year_code
    df["State"] = state_name
    df["District"] = district_name
    df["Block"] = block_name

    # Reorder columns so metadata comes first
    ahead = ["Year","State","District","Block"]
    df = df[ahead + [c for c in df.columns if c not in ahead]]

    return df



states = ['UTTARAKHAND.json5']
for state in states:

    _path_ = Path(state)

    fin_years = {
        "2016-2017": "2016-2017",
        "2017-2018": "2017-2018",
        "2018-2019": "2018-2019",
        "2019-2020": "2019-2020",
        "2020-2021": "2020-2021",
        "2021-2022": "2021-2022",
        "2022-2023": "2022-2023",
        "2023-2024": "2023-2024",
        "2024-2025": "2024-2025",
        "2025-2026": "2025-2026",
        # "2026-2027": "2026-2027",
    }


    config = utils.load_json5(state)
    all_dfs = []
    for year_code in fin_years.keys():
        print(year_code)
        for state_code, state_data in config.items():
            # if state_code != "18":  # Maharashtra only
            #     continue
        
            state_name = state_data["name"]
            print(f"STATE: {state_name}")
            for district_code, district_data in state_data["districts"].items():
                district_name = district_data["name"]
                # if district_name not in select_dists:
                #     continue
                
                print(f">>{district_name}")
                for block_code, block_data in district_data["blocks"].items():
                    # block_name = block_data["name"]
                    block_name = block_data
                    print(f"\t{block_name}")

                    df = fetch_block_level_data(
                        year_code, state_code, district_code,
                        block_code, state_name, district_name, block_name
                    )
                    all_dfs.append(df)
                    time.sleep(.4)

    from datetime import datetime
    today = datetime.now().strftime("%d%m%Y%H%M")
    final_df = pd.concat(all_dfs, ignore_index=True)
    final_df.to_excel(f"{_path_.stem}_{today}.xlsx", index=False)              

	THARALI
>>CHAMPAWAT
	BARAKOT
	CHAMPAWAT
	LOHAGHAT
	PATI
>>DEHRADUN
	CHAKRATA
	DOIWALA
	KALSI
	RAIPUR
	SAHASPUR
	VIKASNAGAR
>>HARIDWAR
	BAHADRABAD
	BHAGWANPUR
	KHANPUR
	LAKSAR
	NARSAN
	ROORKEE
>>NAINITAL
	BETALGHAT
	BHIMTAL
	DHARI
	HALDWANI
	KOTABAG
	OKHALKANDA
	RAMGARH
	RAMNAGAR
>>PAURI GARHWAL
	BIRONKHAL
	DUGGADA
	DWARIKHAL
	EKESHWAR
	KALJIKHAL
	KHIRSU
	KOT
	NAINIDANDA
	PABAU
	PAURI
	POKHRA
	RIKHNIKHAL
	THALISAIN
	YAMKESHWAR
	ZAHRIKHAL
>>PITHORAGARH
	BERINAG
	DHARCHULA
	DIDIHAT
	GANGOLIHAT
	KANALICHINA
	MUNAKOT
	MUNSYARI
	PITHORAGARH
>>RUDRA PRAYAG
	AUGUSTYAMUNI
	JAKHOLI
	UKHIMATH
>>TEHRI GARHWAL
	BHILANGNA
	CHAMBA
	DEOPRAYAG
	JAKHNIDHAR
	JAUNPUR
	KIRTINAGAR
	NARENDRA NAGAR
	PRATAPNAGAR
	THAULDHAR
>>UDAM SINGH NAGAR
	BAJPUR
	GADARPUR
	JASPUR
	KASHIPUR
	KHATIMA
	RUDRAPUR
	SITARGANJ
>>UTTAR KASHI
	BHATWARI
	CHINYALISAUR
	DUNDA
	MORI
	NAUGAON
	PUROLA
2020-2021
STATE: UTTARAKHAND
>>ALMORA
	BHAISIACHHANA
	BHIKIASAIN
	CHAUKHUTIA
	DHAULADEVI
	DWARAHAT
	HAWALBAG
	LAMGARA
	SULT
	SYALDEY
	TAKU